In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from google.colab import drive

drive.mount('/content/drive')

dataset_dir = '/content/drive/MyDrive/ESAM APP/dataset_esam'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False 

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(3, activation='softmax') # 3 kelas: organik, anorganik, b3
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

EPOCHS = 10
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)

save_dir = '/content/drive/MyDrive/ESAM APP/models'
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, 'model_esam.h5'))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(os.path.join(save_dir, 'model_esam.tflite'), 'wb') as f:
    f.write(tflite_model)

print("\nTRAINING SELESAI & MODEL SUDAH TERSIMPAN DI DRIVE!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 12412 images belonging to 3 classes.
Found 3103 images belonging to 3 classes.


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 1017s 3s/step - accuracy: 0.9571 - loss: 0.1386 - val_accuracy: 0.9452 - val_loss: 0.1437
Epoch 2/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 941s 2s/step - accuracy: 0.9811 - loss: 0.0566 - val_accuracy: 0.9307 - val_loss: 0.1765
Epoch 3/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 953s 2s/step - accuracy: 0.9835 - loss: 0.0463 - val_accuracy: 0.9359 - val_loss: 0.1628
Epoch 4/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 961s 2s/step - accuracy: 0.9853 - loss: 0.0425 - val_accuracy: 0.9523 - val_loss: 0.1276
Epoch 5/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 941s 2s/step - accuracy: 0.9872 - loss: 0.0394 - val_accuracy: 0.9497 - val_loss: 0.1300
Epoch 6/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 943s 2s/step - accuracy: 0.9872 - loss: 0.0369 - val_accuracy: 0.9555 - val_loss: 0.1164
Epoch 7/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 934s 2s/step - accuracy: 0.9874 - loss: 0.0364 - val_accuracy: 0.9507 - val_loss: 0.1338
Epoch 8/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 934s 2s/step - accuracy: 0.9878 - loss: 0.0335 - val_acc

Saved artifact at '/tmp/tmplg7iy56f'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_480')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136683314658768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314657808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314658576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314657616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314658960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314658192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314656848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314657040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314656656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136683314659152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1366833146

In [15]:
import numpy as np
from google.colab import files
from tensorflow.keras.preprocessing import image

# 1. Ambil urutan label resmi langsung dari train_generator
# (Mengikuti indeks asli waktu training)
labels_map = dict((v, k) for k, v in train_generator.class_indices.items())
print("Urutan Label Asli Model:", labels_map)

# 2. Upload gambar
uploaded = files.upload()

for filename in uploaded.keys():
    img = image.load_img(filename, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)
    pred_index = np.argmax(predictions[0])

    # Ambil nama kelas sesuai indeks asli
    predicted_class = labels_map[pred_index]
    confidence = np.max(predictions[0]) * 100

    print(f"\n--- HASIL PREDIKSI AKURAT ---")
    print(f"File         : {filename}")
    print(f"Jenis Sampah : {predicted_class.upper()}")
    print(f"Tingkat Keyakinan: {confidence:.2f}%")

Urutan Label Asli Model: {0: 'anorganik', 1: 'b3', 2: 'organik'}


Saving SAMPAH ANORGANIK 2.jpeg to SAMPAH ANORGANIK 2 (1).jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

--- HASIL PREDIKSI AKURAT ---
File         : SAMPAH ANORGANIK 2 (1).jpeg
Jenis Sampah : ANORGANIK
Tingkat Keyakinan: 100.00%


In [ ]:
import os
import shutil

# 1. Kredensial Kaggle
os.environ['KAGGLE_USERNAME'] = 'tegarbei'
os.environ['KAGGLE_KEY'] = 'KGAT_cc68a1a8e53244cb76ff4f3def1160ad'

# 2. Reset folder dataset lama di Drive biar gak nyampur
dataset_dir = '/content/drive/MyDrive/ESAM APP/dataset_esam'
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)

os.makedirs(os.path.join(dataset_dir, 'organik'), exist_ok=True)
os.makedirs(os.path.join(dataset_dir, 'anorganik'), exist_ok=True)
os.makedirs(os.path.join(dataset_dir, 'b3'), exist_ok=True)

# 3. Download dataset bersih langsung ke temp Colab
print("1. Downloading dataset terstruktur dari Kaggle...")
!kaggle datasets download -d mostafaabla/garbage-classification -p /content/temp_garbage --unzip

# 4. Pindahkan sesuai folder resmi Kaggle
base_kaggle = '/content/temp_garbage/garbage_classification'
if not os.path.exists(base_kaggle):
    base_kaggle = '/content/temp_garbage'

print("2. Menyusun folder dataset ESAM...")
for folder in os.listdir(base_kaggle):
    folder_path = os.path.join(base_kaggle, folder)
    if os.path.isdir(folder_path):
        f_name = folder.lower()

        # Kelompokkan folder Kaggle secara TEPAT
        if f_name in ['organic', 'biological', 'food']:
            target = os.path.join(dataset_dir, 'organik')
        elif f_name in ['battery', 'light_bulb', 'e-waste']:
            target = os.path.join(dataset_dir, 'b3')
        else: # cardboard, glass, metal, paper, plastic, trash, clothes, shoes
            target = os.path.join(dataset_dir, 'anorganik')

        for file in os.listdir(folder_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                shutil.copy(os.path.join(folder_path, file), os.path.join(target, f"{f_name}_{file}"))

print("\n--- REKAP DATASET BARU ---")
for cat in ['organik', 'anorganik', 'b3']:
    p = os.path.join(dataset_dir, cat)
    print(f"Total {cat.upper()}: {len(os.listdir(p))} gambar")

1. Downloading dataset terstruktur dari Kaggle...
Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
100% 239M/239M [00:01<00:00, 141MB/s]

2. Menyusun folder dataset ESAM...

--- REKAP DATASET BARU ---
Total ORGANIK: 985 gambar
Total ANORGANIK: 13585 gambar
Total B3: 945 gambar
